In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, TimeSeriesSplit
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, 
    Concatenate, LSTM, Bidirectional,
    Embedding, Flatten, GlobalAveragePooling1D,
    LayerNormalization, MultiHeadAttention
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import random
from datetime import datetime, timedelta


In [2]:

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)
os.environ['PYTHONHASHSEED'] = str(42)

print("Starting the simplified TFT weather forecasting process...")

# Load data
print("Loading data...")
train =  pd.read_csv(r"D:\CVs\New folder\data-crunch-round-1\train.csv")
print(f"Train data shape: {train.shape}")

# Examine the Year, Month, Day columns
print("Year values (first 10):", train['Year'].head(10).tolist())
print("Month values (first 10):", train['Month'].head(10).tolist())
print("Day values (first 10):", train['Day'].head(10).tolist())

# Check if we need to add 2002 to years (instead of 2000)
if train['Year'].min() < 100:
    print("Detected 2-digit years, converting to 4-digit years...")
    # Add 2002 so that year 2 becomes 2004 (a leap year)
    train['Year'] = train['Year'].apply(lambda x: x + 2002 if x < 100 else x)
    
print("Year range after adjustment:", train['Year'].min(), "to", train['Year'].max())

# Create date column with proper formatting and error handling
print("Creating date column with error handling...")
train['date'] = pd.to_datetime({
    'year': train['Year'],
    'month': train['Month'], 
    'day': train['Day']
}, errors='coerce')  # 'coerce' will set invalid dates to NaT

# Check for NaT values
nat_count = train['date'].isna().sum()
if nat_count > 0:
    print(f"WARNING: {nat_count} invalid dates detected and set to NaT")
    # Show some examples of rows with invalid dates
    print("Examples of rows with invalid dates:")
    print(train[train['date'].isna()].head()[['Year', 'Month', 'Day']])
    
    # Drop rows with invalid dates
    train = train.dropna(subset=['date'])
    print(f"Dropped {nat_count} rows with invalid dates. New shape: {train.shape}")

train = train.drop(['Year', 'Month', 'Day'], axis=1)

# Load test data
test = pd.read_csv(r"D:\CVs\New folder\data-crunch-round-1\test.csv")
print(f"Test data shape: {test.shape}")

# Check if we need to add 2002 to years in test data (instead of 2000)
if test['Year'].min() < 100:
    print("Detected 2-digit years in test data, converting to 4-digit years...")
    # Add 2002 so that year 2 becomes 2004 (a leap year)
    test['Year'] = test['Year'].apply(lambda x: x + 2002 if x < 100 else x)

# Join kingdom coordinates from train to test
kingdom_coords = train[['kingdom', 'latitude', 'longitude']].drop_duplicates()
print(f"Found {len(kingdom_coords)} unique kingdoms")

test = test.merge(kingdom_coords, on='kingdom', how='left')

# Create date column for test data
test['date'] = pd.to_datetime({
    'year': test['Year'],
    'month': test['Month'], 
    'day': test['Day']
}, errors='coerce')

# Check for NaT values in test
nat_count = test['date'].isna().sum()
if nat_count > 0:
    print(f"WARNING: {nat_count} invalid dates detected in test data")
    # For this example, we'll need to handle these somehow
    # One approach is to assign a default date
    default_date = pd.Timestamp('2000-01-01')
    test.loc[test['date'].isna(), 'date'] = default_date
    print(f"Assigned default date {default_date} to {nat_count} test rows with invalid dates")

test = test.drop(['Year', 'Month', 'Day'], axis=1)

# Print sample dates to verify
print("Sample train dates:", train['date'].head())
print("Sample test dates:", test['date'].head())

# Function to detect and convert Kelvin to Celsius
def convert_kelvin_to_celsius(df, temp_column='Avg_Temperature'):
    # Kelvin values are typically > 273.15 (0°C in Kelvin)
    # A reasonable threshold might be around 100°C (373.15K)
    kelvin_threshold = 100
    
    # Count potential Kelvin values
    kelvin_count = (df[temp_column] > kelvin_threshold).sum()
    total_count = len(df)
    
    print(f"Detected {kelvin_count} out of {total_count} temperature values potentially in Kelvin")
    
    if kelvin_count > 0:
        # If we have Kelvin values, convert them
        print("Converting Kelvin values to Celsius...")
        df.loc[df[temp_column] > kelvin_threshold, temp_column] = df.loc[df[temp_column] > kelvin_threshold, temp_column] - 273.15
        
        # Verify the conversion
        new_max = df[temp_column].max()
        new_min = df[temp_column].min()
        print(f"After conversion: Temperature range is now {new_min:.2f}°C to {new_max:.2f}°C")
    
    return df

# Apply the conversion to train data
train = convert_kelvin_to_celsius(train)

# Also apply to test data if it contains temperature values
if 'Avg_Temperature' in test.columns and not test['Avg_Temperature'].isna().all():
    test = convert_kelvin_to_celsius(test)


Starting the simplified TFT weather forecasting process...
Loading data...
Train data shape: (84960, 17)
Year values (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Month values (first 10): [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
Day values (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Detected 2-digit years, converting to 4-digit years...
Year range after adjustment: 2003 to 2010
Creating date column with error handling...
Test data shape: (4530, 5)
Detected 2-digit years in test data, converting to 4-digit years...
Found 30 unique kingdoms
Sample train dates: 0   2003-04-01
1   2003-04-01
2   2003-04-01
3   2003-04-01
4   2003-04-01
Name: date, dtype: datetime64[ns]
Sample test dates: 0   2011-01-01
1   2011-01-01
2   2011-01-01
3   2011-01-01
4   2011-01-01
Name: date, dtype: datetime64[ns]
Detected 33984 out of 84960 temperature values potentially in Kelvin
Converting Kelvin values to Celsius...
After conversion: Temperature range is now 17.20°C to 31.50°C


In [3]:

# =================================================================
# CELL 2 — Feature Engineering  (FIXED: real test lag lookback)
# Test rows now get lag features from last 30 days of train data.
# =================================================================

print("Preparing data for feature engineering...")
weather_cols = ['Avg_Temperature', 'Radiation', 'Rain_Amount',
                'Wind_Speed', 'Wind_Direction']

def engineer_time_features(df):
    df = df.copy()
    df['year']             = df['date'].dt.year
    df['month']            = df['date'].dt.month
    df['day']              = df['date'].dt.day
    df['day_of_week']      = df['date'].dt.dayofweek
    df['day_of_year']      = df['date'].dt.dayofyear
    df['quarter']          = df['date'].dt.quarter
    df['is_month_start']   = df['date'].dt.is_month_start.astype(int)
    df['is_month_end']     = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end']   = df['date'].dt.is_quarter_end.astype(int)
    df['is_year_start']    = df['date'].dt.is_year_start.astype(int)
    df['is_year_end']      = df['date'].dt.is_year_end.astype(int)
    df['week_of_year']     = df['date'].dt.isocalendar().week.astype(int)
    df['season']           = df['month'].apply(
        lambda x: 1 if x in [12,1,2] else 2 if x in [3,4,5]
                  else 3 if x in [6,7,8] else 4)
    for period in [7, 30, 365]:
        for k in range(1, 3):
            df[f'sin_{period}_{k}'] = np.sin(2*np.pi*k*df['day_of_year']/period)
            df[f'cos_{period}_{k}'] = np.cos(2*np.pi*k*df['day_of_year']/period)
    return df

def add_derived_targets(df):
    df = df.copy()
    df['Wind_Direction_sin'] = np.sin(np.deg2rad(df['Wind_Direction']))
    df['Wind_Direction_cos'] = np.cos(np.deg2rad(df['Wind_Direction']))
    df['Rain_Amount_log']    = np.log1p(df['Rain_Amount'].clip(lower=0))
    return df

MAX_LAG  = 30
LAG_VALS = [1, 2, 3, 5, 7, 10, 14, 21, 30]

def compute_lag_features(df_sorted):
    lag_cols, all_lag_names = {}, []
    for col in weather_cols:
        for lag in LAG_VALS:
            name = f'{col}_lag_{lag}'
            lag_cols[name] = df_sorted[col].shift(lag)
            all_lag_names.append(name)
    df_sorted = pd.concat(
        [df_sorted, pd.DataFrame(lag_cols, index=df_sorted.index)], axis=1)
    roll_cols = {}
    for lag_name in all_lag_names:
        for w in [3, 7, 14, 30]:
            roll_cols[f'{lag_name}_roll_mean_{w}'] = (
                df_sorted[lag_name].rolling(w, min_periods=1).mean())
            roll_cols[f'{lag_name}_roll_std_{w}'] = (
                df_sorted[lag_name].rolling(w, min_periods=1).std())
    df_sorted = pd.concat(
        [df_sorted, pd.DataFrame(roll_cols, index=df_sorted.index)], axis=1)
    return df_sorted, all_lag_names

print("Applying time and derived features...")
train_fe = engineer_time_features(train.copy())
train_fe = add_derived_targets(train_fe)
train_fe['kingdom_orig'] = train_fe['kingdom']
train_fe['split']        = 'train'

test_fe  = engineer_time_features(test.copy())
test_fe['Wind_Direction_sin'] = np.nan
test_fe['Wind_Direction_cos'] = np.nan
test_fe['Rain_Amount_log']    = np.nan
test_fe['kingdom_orig'] = test_fe['kingdom']
test_fe['split']        = 'test'

print(f"Train rows: {len(train_fe)}  |  Test rows: {len(test_fe)}")
print("\nComputing lag/rolling features by kingdom (test uses real train lookback)...")

kingdoms = train_fe['kingdom'].unique()
train_parts, test_parts = [], []
all_lag_feature_names   = []

start_time = time.time()

for i, kingdom in enumerate(kingdoms):
    if i % 5 == 0:
        print(f"  Processing kingdom {i+1}/{len(kingdoms)}: {kingdom}")

    tr = train_fe[train_fe['kingdom'] == kingdom].copy().sort_values('date')
    tr, lag_names = compute_lag_features(tr)
    train_parts.append(tr)
    all_lag_feature_names.extend(lag_names)

    te      = test_fe[test_fe['kingdom'] == kingdom].copy().sort_values('date')
    # Strip lag/roll cols from tr_tail before concat —
    # compute_lag_features expects raw columns only.
    # tr already has lags computed; we only need the raw weather values
    # as lookback context for the test rows.
    raw_cols_only = [c for c in tr.columns
                     if '_lag_' not in c and '_roll_' not in c]
    tr_tail = tr[raw_cols_only].tail(MAX_LAG).copy()
    tr_tail['split'] = '__context__'

    combined     = pd.concat([tr_tail, te], ignore_index=False).sort_values('date')
    combined, _  = compute_lag_features(combined)
    te_with_lags = combined[combined['split'] == 'test']
    test_parts.append(te_with_lags)

print(f"\nFeature engineering completed in {time.time()-start_time:.2f}s")
print(f"Unique lag feature names: {len(set(all_lag_feature_names))}")

train_prepped_raw = pd.concat(train_parts, ignore_index=True)
test_prepped_raw  = pd.concat(test_parts,  ignore_index=True)

sample_lag     = 'Avg_Temperature_lag_1'
test_zero_pct  = (test_prepped_raw[sample_lag] == 0).mean() * 100
train_zero_pct = (train_prepped_raw[sample_lag] == 0).mean() * 100
print(f"\nZero-value check on '{sample_lag}':")
print(f"  Train: {train_zero_pct:.1f}% zeros  (expected ~0%)")
print(f"  Test:  {test_zero_pct:.1f}% zeros   (should now be ~0%, was 100%)")

max_date = train_prepped_raw['date'].max()
train_prepped_raw['time_weight'] = train_prepped_raw['date'].apply(
    lambda x: 1 / (1 + (max_date - x).days / 365))
test_prepped_raw['time_weight'] = 1.0

print(f"\nFinal train shape: {train_prepped_raw.shape}")
print(f"Final test shape:  {test_prepped_raw.shape}")

train_prepped = train_prepped_raw.copy()
test_prepped  = test_prepped_raw.copy()


Preparing data for feature engineering...
Applying time and derived features...
Train rows: 84960  |  Test rows: 4530

Computing lag/rolling features by kingdom (test uses real train lookback)...
  Processing kingdom 1/30: Arcadia
  Processing kingdom 6/30: Eden
  Processing kingdom 11/30: Krypton
  Processing kingdom 16/30: Neo-City
  Processing kingdom 21/30: Rapture
  Processing kingdom 26/30: Solstice

Feature engineering completed in 4.13s
Unique lag feature names: 45

Zero-value check on 'Avg_Temperature_lag_1':
  Train: 0.0% zeros  (expected ~0%)
  Test:  0.0% zeros   (should now be ~0%, was 100%)

Final train shape: (84960, 452)
Final test shape:  (4530, 452)


In [4]:

# =================================================================
# CELL 3 — Preprocessing  (FIXED: NaN diagnostic + clean fillna)
# =================================================================

print("Handling outliers in target variables...")
for col in weather_cols:
    Q1 = train_prepped[col].quantile(0.01)
    Q3 = train_prepped[col].quantile(0.99)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = train_prepped[
        (train_prepped[col] < lower_bound) | (train_prepped[col] > upper_bound)]
    print(f"Found {len(outliers)} outliers in {col} ({len(outliers)/len(train_prepped)*100:.2f}%)")
    print(f"  Range before: {train_prepped[col].min():.2f} to {train_prepped[col].max():.2f}")
    train_prepped[col] = train_prepped[col].clip(lower_bound, upper_bound)
    print(f"  Range after:  {train_prepped[col].min():.2f} to {train_prepped[col].max():.2f}")

lag_roll_cols = [c for c in train_prepped.columns if '_lag_' in c or '_roll_' in c]

test_nan_pct = test_prepped[lag_roll_cols].isna().mean().mean() * 100
print(f"\nTest lag/roll NaN % BEFORE fillna: {test_nan_pct:.1f}%")
print("  (After Cell 2 fix this should be <5%, not 100%)")

train_prepped[lag_roll_cols] = train_prepped[lag_roll_cols].fillna(0)
test_prepped[lag_roll_cols]  = test_prepped[lag_roll_cols].fillna(0)

numeric_cols = train_prepped.select_dtypes(include=['int64','float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in
                weather_cols + ['ID','time_weight',
                'Wind_Direction_sin','Wind_Direction_cos','Rain_Amount_log']
                + lag_roll_cols + ['date']]
print(f"\nNumeric cols for imputation ({len(numeric_cols)}): {numeric_cols[:8]}...")

print("\nApplying kingdom-wise KNN imputation on train...")
imputed_parts = []
for kingdom in train_prepped['kingdom_orig'].unique():
    k_mask = train_prepped['kingdom_orig'] == kingdom
    k_data = train_prepped.loc[k_mask, numeric_cols]
    imp    = KNNImputer(n_neighbors=5, weights='distance')
    k_imp  = pd.DataFrame(imp.fit_transform(k_data),
                          columns=numeric_cols, index=k_data.index)
    imputed_parts.append(k_imp)

X_train_imputed = pd.concat(imputed_parts).reindex(train_prepped.index)
train_prepped[numeric_cols] = X_train_imputed

print("Applying global KNN imputation on test...")
global_imputer = KNNImputer(n_neighbors=5, weights='distance')
global_imputer.fit(train_prepped[numeric_cols])
test_prepped[numeric_cols] = pd.DataFrame(
    global_imputer.transform(test_prepped[numeric_cols]),
    columns=numeric_cols, index=test_prepped.index)

print("One-hot encoding categorical features...")
train_prepped = pd.get_dummies(train_prepped, columns=['kingdom'], drop_first=False)
test_prepped  = pd.get_dummies(test_prepped,  columns=['kingdom'], drop_first=False)

for col in train_prepped.columns:
    if col not in test_prepped.columns:
        test_prepped[col] = 0
test_prepped = test_prepped[train_prepped.columns]

print("Applying feature scaling...")
scaler = StandardScaler()
train_prepped[numeric_cols] = scaler.fit_transform(train_prepped[numeric_cols])
test_prepped[numeric_cols]  = scaler.transform(test_prepped[numeric_cols])

print(f"\nFinal train shape: {train_prepped.shape}")
print(f"Final test shape:  {test_prepped.shape}")


Handling outliers in target variables...
Found 0 outliers in Avg_Temperature (0.00%)
  Range before: 17.20 to 31.50
  Range after:  17.20 to 31.50
Found 0 outliers in Radiation (0.00%)
  Range before: 3.19 to 30.10
  Range after:  3.19 to 30.10
Found 51 outliers in Rain_Amount (0.06%)
  Range before: 0.00 to 440.44
  Range after:  0.00 to 152.10
Found 0 outliers in Wind_Speed (0.00%)
  Range before: 2.30 to 50.20
  Range after:  2.30 to 50.20
Found 0 outliers in Wind_Direction (0.00%)
  Range before: 0.00 to 359.00
  Range after:  0.00 to 359.00

Test lag/roll NaN % BEFORE fillna: 86.1%
  (After Cell 2 fix this should be <5%, not 100%)

Numeric cols for imputation (27): ['latitude', 'longitude', 'Avg_Feels_Like_Temperature', 'Temperature_Range', 'Feels_Like_Temperature_Range', 'Rain_Duration', 'Evapotranspiration', 'is_month_start']...

Applying kingdom-wise KNN imputation on train...
Applying global KNN imputation on test...
One-hot encoding categorical features...
Applying feature sc

In [5]:
print("\n" + "="*50)
print("IMPLEMENTING ATTENTION-BASED MULTI-OUTPUT FORECASTER")
print("="*50)

y_cols    = ['Avg_Temperature', 'Radiation', 'Rain_Amount_log',
             'Wind_Speed', 'Wind_Direction_sin', 'Wind_Direction_cos']
final_cols = ['Avg_Temperature', 'Radiation', 'Rain_Amount',
              'Wind_Speed', 'Wind_Direction']

exclude_cols = weather_cols + y_cols + ['ID', 'date', 'time_weight', 'kingdom_orig', 'split']
X_cols = [col for col in train_prepped.columns if col not in exclude_cols]

# ── KINGDOM-WISE TRAIN/VAL SPLIT ─────────────────────────────────
print("Applying kingdom-wise 80/20 time split...")
train_idx = []
val_idx   = []

for kingdom in train_prepped['kingdom_orig'].unique():
    k_data   = train_prepped[train_prepped['kingdom_orig'] == kingdom].sort_values('date')
    cutoff   = k_data['date'].quantile(0.8)
    train_idx.extend(k_data[k_data['date'] <  cutoff].index.tolist())
    val_idx.extend(  k_data[k_data['date'] >= cutoff].index.tolist())

X_train = train_prepped.loc[train_idx, X_cols]
X_val   = train_prepped.loc[val_idx,   X_cols]
y_train = train_prepped.loc[train_idx, y_cols]
y_val   = train_prepped.loc[val_idx,   y_cols]

# Store kingdom names for val set before dropping
val_kingdom_names = train_prepped.loc[val_idx, 'kingdom_orig'].values

print(f"Kingdom-wise split done.")
print(f"Train rows: {len(X_train)}, Val rows: {len(X_val)}")

# Sample weights
train_weights = train_prepped.loc[train_idx, 'time_weight'].values

# Test data
X_test   = test_prepped[X_cols]
test_ids = test_prepped['ID'].values

print(f"Training data shape:   {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Test data shape:       {X_test.shape}")


IMPLEMENTING ATTENTION-BASED MULTI-OUTPUT FORECASTER
Applying kingdom-wise 80/20 time split...
Kingdom-wise split done.
Train rows: 67950, Val rows: 17010
Training data shape:   (67950, 468)
Validation data shape: (17010, 468)
Test data shape:       (4530, 468)


In [6]:
# =============================================================
# UPGRADED ATTENTION FORECASTER  v2
# Changes vs original:
#   1. Structured sequence — lag features grouped by variable
#      into real (seq_len, d_model) input, not a reshape hack
#   2. Kingdom embedding — learned 8-dim vector, not one-hot only
#   3. Stacked transformer blocks (n_blocks=2 default)
#   4. Rain two-stage gate inside the model
#      (sigmoid gate × relu magnitude)
#   5. Per-target output heads after a shared trunk
#   6. Same combined sMAPE + angular loss you already had
# =============================================================

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    Flatten, LayerNormalization, MultiHeadAttention,
    Concatenate, Add, Embedding, Reshape, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


# ── helper: one transformer encoder block ────────────────────────
def transformer_block(x, num_heads, ff_dim, dropout_rate, name_prefix):
    """Pre-norm transformer block with residual connections."""
    d_model = x.shape[-1]

    # Self-attention
    x_norm = LayerNormalization(name=f"{name_prefix}_ln1")(x)
    attn   = MultiHeadAttention(
                num_heads=num_heads,
                key_dim=d_model // num_heads,
                dropout=dropout_rate,
                name=f"{name_prefix}_mha"
             )(x_norm, x_norm)
    x = Add(name=f"{name_prefix}_add1")([x, attn])

    # Feed-forward
    x_norm = LayerNormalization(name=f"{name_prefix}_ln2")(x)
    ff     = Dense(ff_dim,   activation="gelu", name=f"{name_prefix}_ff1")(x_norm)
    ff     = Dropout(dropout_rate,              name=f"{name_prefix}_drop1")(ff)
    ff     = Dense(d_model,                    name=f"{name_prefix}_ff2")(ff)
    ff     = Dropout(dropout_rate,              name=f"{name_prefix}_drop2")(ff)
    x = Add(name=f"{name_prefix}_add2")([x, ff])
    return x


def create_attention_forecaster_v2(
    input_shape,          # total flat feature count  (e.g. 75-ish)
    num_kingdoms=30,      # for embedding lookup
    kingdom_embed_dim=8,  # learned kingdom embedding size
    d_model=64,           # transformer hidden dim
    num_heads=4,
    ff_dim=128,           # feed-forward expansion inside transformer
    n_blocks=2,           # number of stacked transformer blocks
    dropout_rate=0.2,
    learning_rate=0.001,
    # Feature-group sizes — adjust to match your actual X_cols layout.
    # The code auto-groups; these are used only if you pass group_sizes.
    seq_len=5,            # number of "tokens" (weather variable groups)
):
    """
    Structured Attention Forecaster.

    Input layout expected in X_cols (set in Cell 4):
      - One-hot kingdom columns  (30 cols, last ones)
      - Lag/rolling features grouped by the 5 weather variables
      - Time features

    Architecture:
      flat_features  →  Dense projection  →  (seq_len, d_model) tokens
      kingdom_id     →  Embedding(30, 8)  →  broadcast + add to tokens
      tokens         →  N × TransformerBlock
                     →  GlobalAvgPool
                     →  shared trunk
                     →  per-target output heads
    """

    # ── Two inputs: feature vector + kingdom integer id ───────────
    feat_input    = Input(shape=(input_shape,),  name="features")
    kingdom_input = Input(shape=(1,),            name="kingdom_id",
                          dtype="int32")

    # ── Kingdom embedding ─────────────────────────────────────────
    k_embed = Embedding(num_kingdoms, kingdom_embed_dim,
                        name="kingdom_embed")(kingdom_input)   # (B,1,8)
    k_embed = Flatten(name="k_flat")(k_embed)                  # (B,8)
    k_proj  = Dense(d_model, activation="linear",
                    name="k_proj")(k_embed)                    # (B,d_model)

    # ── Encode flat features → sequence of tokens ────────────────
    # Project to (seq_len * d_model), then reshape to (seq_len, d_model).
    # Each "token" roughly corresponds to one weather-variable group.
    proj = Dense(seq_len * d_model, activation="relu",
                 name="feat_proj")(feat_input)
    proj = BatchNormalization(name="feat_bn")(proj)
    proj = Dropout(dropout_rate, name="feat_drop")(proj)

    tokens = Reshape((seq_len, d_model), name="reshape_tokens")(proj)

    # Add kingdom embedding to every token (broadcast over seq dim)
    k_broadcast = Reshape((1, d_model), name="k_broadcast")(k_proj)
    tokens = Add(name="kingdom_add")([tokens, k_broadcast])

    # ── Stacked transformer blocks ────────────────────────────────
    for b in range(n_blocks):
        tokens = transformer_block(
            tokens,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate,
            name_prefix=f"blk{b}"
        )

    # ── Aggregate sequence → single vector ───────────────────────
    # Global average pool across the seq_len dimension
    pooled = tf.keras.layers.GlobalAveragePooling1D(name="gap")(tokens)

    # ── Shared trunk ──────────────────────────────────────────────
    trunk = Dense(d_model * 2, activation="gelu", name="trunk1")(pooled)
    trunk = BatchNormalization(name="trunk_bn")(trunk)
    trunk = Dropout(dropout_rate, name="trunk_drop")(trunk)
    trunk = Dense(d_model,       activation="gelu", name="trunk2")(trunk)

    # ── Per-target output heads (physics-constrained) ─────────────
    # Temperature: linear  (can be negative)
    temp_out  = Dense(1, activation="linear", name="temp")(trunk)

    # Radiation: relu  (non-negative)
    rad_out   = Dense(1, activation="relu",   name="radiation")(trunk)

    # Rain: two-stage gate × magnitude  (zero-inflated)
    rain_gate = Dense(1, activation="sigmoid", name="rain_gate")(trunk)
    rain_mag  = Dense(1, activation="relu",    name="rain_mag")(trunk)
    rain_out  = tf.keras.layers.Multiply(      name="rain")([rain_gate, rain_mag])

    # Wind speed: relu
    wspd_out  = Dense(1, activation="relu",   name="wind_speed")(trunk)

    # Wind direction: sin/cos with tanh
    wsin_out  = Dense(1, activation="tanh",   name="wind_sin")(trunk)
    wcos_out  = Dense(1, activation="tanh",   name="wind_cos")(trunk)

    outputs = Concatenate(name="outputs")([
        temp_out, rad_out, rain_out,
        wspd_out, wsin_out, wcos_out
    ])

    model = Model(inputs=[feat_input, kingdom_input], outputs=outputs,
                  name="HarvestonForecaster_v2")

    # ── Loss: sMAPE for first 4 + angular for wind dir ────────────
    def combined_loss(y_true, y_pred):
        eps = tf.keras.backend.epsilon()

        y_true_main = y_true[:, :4]
        y_pred_main = y_pred[:, :4]
        sum_abs     = tf.abs(y_true_main) + tf.abs(y_pred_main) + eps
        smape_loss  = tf.reduce_mean(200.0 * tf.abs(y_pred_main - y_true_main) / sum_abs)

        sin_t, cos_t = y_true[:, 4], y_true[:, 5]
        sin_p, cos_p = y_pred[:, 4], y_pred[:, 5]
        ang_loss     = tf.reduce_mean(1.0 - (cos_t*cos_p + sin_t*sin_p))

        return smape_loss + 5.0 * ang_loss   # 2.0 -> 5.0: force model to learn wind direction

    model.compile(
        optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss=combined_loss,
        metrics=["mae"]
    )
    return model


# ── Quick sanity check ────────────────────────────────────────────
print("Model architecture defined.")
print("  - Kingdom embedding: 30 kingdoms → 8-dim learned vector")
print("  - Sequence tokens:   5 tokens × 64-dim (one per weather var group)")
print(f"  - Transformer blocks: 2 stacked with {4} heads each")
print("  - Rain two-stage gate: sigmoid(gate) × relu(magnitude)")
print("  - Physics constraints: linear/relu/relu/relu/tanh/tanh per output")


Model architecture defined.
  - Kingdom embedding: 30 kingdoms → 8-dim learned vector
  - Sequence tokens:   5 tokens × 64-dim (one per weather var group)
  - Transformer blocks: 2 stacked with 4 heads each
  - Rain two-stage gate: sigmoid(gate) × relu(magnitude)
  - Physics constraints: linear/relu/relu/relu/tanh/tanh per output


In [7]:
# ── Prepare kingdom integer id for embedding ─────────────────────
# We need an integer kingdom index alongside the float feature matrix.

# Build a kingdom → int mapping from the original kingdom_orig column
kingdom_list = sorted(train_prepped["kingdom_orig"].unique())
kingdom2idx  = {k: i for i, k in enumerate(kingdom_list)}
NUM_KINGDOMS = len(kingdom_list)
print(f"Kingdom mapping built: {NUM_KINGDOMS} kingdoms")

# Integer arrays for train, val, test
train_kingdom_ids = train_prepped.loc[train_idx, "kingdom_orig"].map(kingdom2idx).values.reshape(-1,1).astype("int32")
val_kingdom_ids   = train_prepped.loc[val_idx,   "kingdom_orig"].map(kingdom2idx).values.reshape(-1,1).astype("int32")
test_kingdom_ids  = test_prepped["kingdom_orig"].map(kingdom2idx).values.reshape(-1,1).astype("int32")

print(f"train_kingdom_ids shape: {train_kingdom_ids.shape}")
print(f"val_kingdom_ids   shape: {val_kingdom_ids.shape}")
print(f"test_kingdom_ids  shape: {test_kingdom_ids.shape}")


Kingdom mapping built: 30 kingdoms
train_kingdom_ids shape: (67950, 1)
val_kingdom_ids   shape: (17010, 1)
test_kingdom_ids  shape: (4530, 1)


In [8]:
# ── Build and train the upgraded model ───────────────────────────

params = {
    "d_model":       64,
    "num_heads":     4,
    "ff_dim":        128,
    "n_blocks":      2,
    "seq_len":       5,      # one token per weather-variable group
    "kingdom_embed": 8,
    "dropout_rate":  0.25,
    "learning_rate": 0.001,
    "batch_size":    128,
}

model = create_attention_forecaster_v2(
    input_shape     = X_train.shape[1],
    num_kingdoms    = NUM_KINGDOMS,
    kingdom_embed_dim = params["kingdom_embed"],
    d_model         = params["d_model"],
    num_heads       = params["num_heads"],
    ff_dim          = params["ff_dim"],
    n_blocks        = params["n_blocks"],
    seq_len         = params["seq_len"],
    dropout_rate    = params["dropout_rate"],
    learning_rate   = params["learning_rate"],
)
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_harveston_v2.keras",
        monitor="val_loss", save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5,
        min_lr=1e-6, verbose=1
    ),
]

X_train_arr = np.nan_to_num(X_train.values.astype("float32"),  nan=0.0, posinf=0.0, neginf=0.0)
X_val_arr   = np.nan_to_num(X_val.values.astype("float32"),    nan=0.0, posinf=0.0, neginf=0.0)
y_train_arr = np.nan_to_num(y_train.values.astype("float32"),  nan=0.0, posinf=0.0, neginf=0.0)
y_val_arr   = np.nan_to_num(y_val.values.astype("float32"),    nan=0.0, posinf=0.0, neginf=0.0)

print(f"X_train: {X_train_arr.shape}  y_train: {y_train_arr.shape}")
print(f"X_val:   {X_val_arr.shape}    y_val:   {y_val_arr.shape}")

history = model.fit(
    [X_train_arr, train_kingdom_ids],
    y_train_arr,
    validation_data=([X_val_arr, val_kingdom_ids], y_val_arr),
    epochs       = 100,
    batch_size   = params["batch_size"],
    sample_weight= train_weights_array if "train_weights_array" in dir() else None,
    callbacks    = callbacks,
    verbose      = 1,
)

# Plot training curve
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(history.history["loss"],     label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.title("Training Loss (combined sMAPE + angular)")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.tight_layout()
plt.savefig("training_loss_v2.png")
plt.close()
print("Training complete. Loss curve saved.")


Model: "HarvestonForecaster_v2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ features            │ (None, 468)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kingdom_id          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_proj (Dense)   │ (None, 320)       │    150,080 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kingdom_embed       │ (None, 1, 8)      │        240 │ kingdom_id[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_bn             │ (None, 320)       │      1,280 │ feat_proj[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ k_flat (Flatten)    │ (None, 8)         │          0 │ kingdom_embed[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_drop (Dropout) │ (None, 320)       │          0 │ feat_bn[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ k_proj (Dense)      │ (None, 64)        │        576 │ k_flat[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_tokens      │ (None, 5, 64)     │          0 │ feat_drop[0][0]   │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ k_broadcast         │ (None, 1, 64)     │          0 │ k_proj[0][0]      │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kingdom_add (Add)   │ (None, 5, 64)     │          0 │ reshape_tokens[0… │
│                     │                   │            │ k_broadcast[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_ln1            │ (None, 5, 64)     │        128 │ kingdom_add[0][0] │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_mha            │ (None, 5, 64)     │     16,640 │ blk0_ln1[0][0],   │
│ (MultiHeadAttentio… │                   │            │ blk0_ln1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_add1 (Add)     │ (None, 5, 64)     │          0 │ kingdom_add[0][0… │
│                     │                   │            │ blk0_mha[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_ln2            │ (None, 5, 64)     │        128 │ blk0_add1[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_ff1 (Dense)    │ (None, 5, 128)    │      8,320 │ blk0_ln2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_drop1          │ (None, 5, 128)    │          0 │ blk0_ff1[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_ff2 (Dense)    │ (None, 5, 64)     │      8,256 │ blk0_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ blk0_drop2          │ (None, 5, 64)     │          0 │ blk0_ff2[0][0]  

 Total params: 236,663 (924.46 KB)

 Trainable params: 235,767 (920.96 KB)

 Non-trainable params: 896 (3.50 KB)

X_train: (67950, 468)  y_train: (67950, 6)
X_val:   (17010, 468)    y_val:   (17010, 6)
Epoch 1/100
529/531 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 72.2894 - mae: 4.4446
Epoch 1: val_loss improved from None to 30.55945, saving model to best_harveston_v2.keras

Epoch 1: finished saving model to best_harveston_v2.keras
531/531 ━━━━━━━━━━━━━━━━━━━━ 22s 22ms/step - loss: 45.6864 - mae: 2.6489 - val_loss: 30.5595 - val_mae: 1.7868 - learning_rate: 0.0010
Epoch 2/100
529/531 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 29.6317 - mae: 1.5866
Epoch 2: val_loss improved from 30.55945 to 25.96790, saving model to best_harveston_v2.keras

Epoch 2: finished saving model to best_harveston_v2.keras
531/531 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - loss: 29.1929 - mae: 1.5459 - val_loss: 25.9679 - val_mae: 1.4307 - learning_rate: 0.0010
Epoch 3/100
531/531 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 27.7089 - mae: 1.4368
Epoch 3: val_loss improved from 25.96790 to 24.10781, saving model to best_harveston_v2.

In [12]:
# ── Validation predictions & metrics ─────────────────────────────

def calculate_smape(actual, predicted):
    mask = (np.abs(actual) + np.abs(predicted)) > 0
    if mask.sum() == 0:
        return 0.0
    return 100 * np.mean(
        2 * np.abs(predicted[mask] - actual[mask]) /
        (np.abs(actual[mask]) + np.abs(predicted[mask]))
    )

y_cols     = ["Avg_Temperature", "Radiation", "Rain_Amount_log",
              "Wind_Speed", "Wind_Direction_sin", "Wind_Direction_cos"]
final_cols = ["Avg_Temperature", "Radiation", "Rain_Amount",
              "Wind_Speed", "Wind_Direction"]

try:
    val_pred    = model.predict([X_val_arr, val_kingdom_ids], verbose=1)
    y_val_array = y_val_arr
    idx         = {col: i for i, col in enumerate(y_cols)}

    real_targets, real_preds = {}, {}

    real_targets["Avg_Temperature"] = y_val_array[:, idx["Avg_Temperature"]]
    real_preds["Avg_Temperature"]   = val_pred[:,   idx["Avg_Temperature"]]

    real_targets["Radiation"]       = y_val_array[:, idx["Radiation"]]
    real_preds["Radiation"]         = np.clip(val_pred[:, idx["Radiation"]], 0, None)

    real_targets["Rain_Amount"]     = np.expm1(y_val_array[:, idx["Rain_Amount_log"]])
    rain_val_raw = np.clip(np.expm1(val_pred[:, idx["Rain_Amount_log"]]), 0, None)
    RAIN_THRESHOLD = 0.5  # dry-day threshold: reduces SMAPE on zero-rain days
    real_preds["Rain_Amount"] = np.where(rain_val_raw < RAIN_THRESHOLD, 0.0, rain_val_raw)

    real_targets["Wind_Speed"]      = y_val_array[:, idx["Wind_Speed"]]
    real_preds["Wind_Speed"]        = np.clip(val_pred[:, idx["Wind_Speed"]], 0, None)

    real_targets["Wind_Direction"]  = np.degrees(np.arctan2(
        y_val_array[:, idx["Wind_Direction_sin"]],
        y_val_array[:, idx["Wind_Direction_cos"]])) % 360
    real_preds["Wind_Direction"]    = np.degrees(np.arctan2(
        val_pred[:,   idx["Wind_Direction_sin"]],
        val_pred[:,   idx["Wind_Direction_cos"]])) % 360

    # Results df
    results_df             = pd.DataFrame()
    results_df["kingdom"]  = val_kingdom_names
    for col in final_cols:
        results_df[f"{col}_actual"]   = real_targets[col]
        results_df[f"{col}_pred"]     = real_preds[col]
        results_df[f"{col}_residual"] = real_targets[col] - real_preds[col]

    # sMAPE by target
    print("\n" + "="*55)
    print("SMAPE BY TARGET")
    print("="*55)
    smape_scores = {}
    for col in final_cols:
        s = calculate_smape(results_df[f"{col}_actual"].values,
                            results_df[f"{col}_pred"].values)
        smape_scores[col] = s
        print(f"  {col:<22} {s:>7.2f}%")
    overall_smape = np.mean(list(smape_scores.values()))
    print("-"*55)
    print(f"  {'Overall sMAPE':<22} {overall_smape:>7.2f}%")

    # Kingdom-level sMAPE
    print("\n" + "="*55)
    print("SMAPE BY KINGDOM")
    print("="*55)
    kingdom_records = []
    for kingdom in sorted(results_df["kingdom"].unique()):
        k   = results_df[results_df["kingdom"] == kingdom]
        row = {"Kingdom": kingdom}
        for col in final_cols:
            row[col] = round(calculate_smape(
                k[f"{col}_actual"].values, k[f"{col}_pred"].values), 2)
        row["Overall"] = round(np.mean([row[c] for c in final_cols]), 2)
        kingdom_records.append(row)

    kingdom_df = (pd.DataFrame(kingdom_records)
                  .sort_values("Overall", ascending=False)
                  .reset_index(drop=True))
    print(kingdom_df.to_string(index=False))

    panel_overall = kingdom_df["Overall"].mean()
    print(f"\n  Panel Overall sMAPE: {panel_overall:.2f}%")

    # Circular wind error
    angle_diff  = results_df["Wind_Direction_actual"] - results_df["Wind_Direction_pred"]
    angular_err = np.abs((angle_diff + 180) % 360 - 180)
    mask        = ~(np.isnan(angular_err) | np.isinf(angular_err))
    print(f"\n  Wind Mean Angular Error:   {angular_err[mask].mean():.2f}°")
    print(f"  Wind Median Angular Error: {angular_err[mask].median():.2f}°")

    # Bias correction
    print("\nBias correction factors:")
    validation_errors = {}
    for i, col in enumerate(y_cols):
        a, p = y_val_array[:, i], val_pred[:, i]
        mask = ~(np.isnan(a)|np.isnan(p)|np.isinf(a)|np.isinf(p))
        validation_errors[col] = float(np.mean(a[mask] - p[mask])) if mask.sum() > 0 else 0.0
        print(f"  {col:<35} {validation_errors[col]:.4f}")

    # sMAPE bar chart
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,5))
    bars = plt.bar(smape_scores.keys(), smape_scores.values(), color="steelblue")
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x()+bar.get_width()/2., h+0.3,
                 f"{h:.2f}%", ha="center", va="bottom", fontsize=9)
    plt.title("Harveston Attention v2 — sMAPE by Target")
    plt.ylabel("sMAPE (%)")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("smape_by_target_v2.png"); plt.close()

    # Kingdom heatmap
    plt.figure(figsize=(14, max(6, len(kingdom_df)*0.4)))
    import seaborn as sns
    sns.heatmap(kingdom_df.set_index("Kingdom")[final_cols],
                annot=True, fmt=".1f", cmap="RdYlGn_r",
                linewidths=0.5, cbar_kws={"label": "sMAPE (%)"})
    plt.title("sMAPE by Kingdom and Target  (v2)")
    plt.tight_layout()
    plt.savefig("kingdom_smape_heatmap_v2.png"); plt.close()
    print("\nCharts saved.")

except Exception as e:
    import traceback; traceback.print_exc()
    validation_errors = {col: 0.0 for col in y_cols}
    smape_scores      = {col: 0.0 for col in final_cols}
    overall_smape     = 0.0
    kingdom_df        = pd.DataFrame()


532/532 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

SMAPE BY TARGET
  Avg_Temperature           1.41%
  Radiation                 4.50%
  Rain_Amount              58.04%
  Wind_Speed               15.27%
  Wind_Direction           28.65%
-------------------------------------------------------
  Overall sMAPE            21.57%

SMAPE BY KINGDOM
     Kingdom  Avg_Temperature  Radiation  Rain_Amount  Wind_Speed  Wind_Direction   Overall
     Valyria             1.66       6.30    82.010002   18.520000       45.709999 30.840000
      Mordor             1.23       5.47    83.570000   15.860000       25.480000 26.320000
    Solstice             1.59       6.21    76.790001   14.410000       32.119999 26.219999
  Winterfell             2.40       4.73    77.019997   19.700001       25.200001 25.809999
    Sunspear             1.49       6.62    70.779999   14.690000       32.750000 25.270000
       Dorne             1.34       5.82    70.800003   13.610000       30.940001 24.500000
   Rivendell        

In [10]:
# ── Generate test predictions & submission ────────────────────────

try:
    X_test_arr = np.nan_to_num(
        X_test.values.astype("float32"), nan=0.0, posinf=0.0, neginf=0.0)

    predictions = model.predict([X_test_arr, test_kingdom_ids], verbose=1)

    # Apply bias correction
    print("Applying bias correction...")
    for i, col in enumerate(y_cols):
        predictions[:, i] += validation_errors[col]

    # Inverse transform
    idx = {col: i for i, col in enumerate(y_cols)}
    test_real = {}
    test_real["Avg_Temperature"] = predictions[:, idx["Avg_Temperature"]]
    test_real["Radiation"]       = np.clip(predictions[:, idx["Radiation"]], 0, None)
    rain_raw = np.clip(np.expm1(predictions[:, idx["Rain_Amount_log"]]), 0, None)
    RAIN_THRESHOLD = 0.5
    test_real["Rain_Amount"] = np.where(rain_raw < RAIN_THRESHOLD, 0.0, rain_raw)
    test_real["Wind_Speed"]      = np.clip(predictions[:, idx["Wind_Speed"]], 0, None)
    test_real["Wind_Direction"]  = np.degrees(np.arctan2(
        predictions[:, idx["Wind_Direction_sin"]],
        predictions[:, idx["Wind_Direction_cos"]])) % 360

    pred_df = pd.DataFrame({"ID": test_ids})
    for col in final_cols:
        pred_df[col] = test_real[col]

    print("\nSample predictions:")
    print(pred_df.head())

    output_file = "Harveston_v2_Predictions.csv"
    pred_df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}  ({os.path.getsize(output_file)/1024:.2f} KB)")

    import joblib
    model.save("harveston_v2_model.keras")
    joblib.dump(scaler,            "harveston_v2_scaler.joblib")
    joblib.dump(numeric_cols,      "harveston_v2_numeric_cols.joblib")
    joblib.dump(X_cols,            "harveston_v2_X_cols.joblib")
    joblib.dump(validation_errors, "harveston_v2_bias.joblib")
    joblib.dump(kingdom2idx,       "harveston_v2_kingdom2idx.joblib")
    print("All artifacts saved.")

except Exception as e:
    import traceback; traceback.print_exc()


142/142 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Applying bias correction...

Sample predictions:
      ID  Avg_Temperature  Radiation  Rain_Amount  Wind_Speed  Wind_Direction
0  84961        24.901236  22.349709          0.0   12.615623       54.599529
1  84991        22.615562  20.228794          0.0    7.371173       47.261040
2  85021        22.725401  23.604836          0.0   10.962593       47.261040
3  85051        22.766808  25.137524          0.0   11.976233       47.261040
4  85081        22.739695  26.755142          0.0   10.825808       47.261040
Saved: Harveston_v2_Predictions.csv  (236.28 KB)
All artifacts saved.


In [11]:
print("\n" + "="*55)
print("RESIDUAL ANALYSIS")
print("="*55)

try:
    for col in final_cols:
        plt.figure(figsize=(16, 10))

        actual   = results_df[f'{col}_actual']
        pred     = results_df[f'{col}_pred']
        residual = results_df[f'{col}_residual']

        mask = ~(np.isnan(actual)   | np.isnan(pred)   | np.isnan(residual) |
                 np.isinf(actual)   | np.isinf(pred)   | np.isinf(residual))

        actual   = actual[mask].values
        pred     = pred[mask].values
        residual = residual[mask].values

        # Plot 1 — Actual vs Predicted
        plt.subplot(2, 2, 1)
        plt.scatter(actual, pred, alpha=0.4, s=10)
        mn, mx = min(actual.min(), pred.min()), max(actual.max(), pred.max())
        plt.plot([mn, mx], [mn, mx], 'r--')
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title(f'Actual vs Predicted: {col}')
        corr = np.corrcoef(actual, pred)[0, 1]
        plt.annotate(f'r = {corr:.4f}',
                     xy=(0.05, 0.93), xycoords='axes fraction',
                     bbox=dict(boxstyle='round', fc='white', alpha=0.7))

        # Plot 2 — Residuals vs Predicted
        plt.subplot(2, 2, 2)
        plt.scatter(pred, residual, alpha=0.4, s=10)
        plt.axhline(0, color='r', linestyle='--')
        plt.xlabel('Predicted')
        plt.ylabel('Residuals')
        plt.title(f'Residuals vs Predicted: {col}')
        plt.annotate(f'Mean={residual.mean():.3f}\nStd={residual.std():.3f}',
                     xy=(0.05, 0.88), xycoords='axes fraction',
                     bbox=dict(boxstyle='round', fc='white', alpha=0.7))

        # Plot 3 — Residual distribution
        plt.subplot(2, 2, 3)
        sns.histplot(residual, kde=True)
        plt.xlabel('Residual')
        plt.title(f'Residual Distribution: {col}')

        # Plot 4 — Q-Q plot
        plt.subplot(2, 2, 4)
        from scipy import stats
        stats.probplot(residual, dist='norm', plot=plt)
        plt.title(f'Q-Q Plot: {col}')

        plt.tight_layout()
        plt.savefig(f'residual_analysis_{col}.png')
        plt.close()
        print(f"  Saved residual plot: {col}")

    # Kingdom residual boxplots
    for col in final_cols:
        plt.figure(figsize=(max(12, len(kingdom_df)*0.6), 6))
        sns.boxplot(x='kingdom', y=f'{col}_residual', data=results_df)
        plt.title(f'Residual by Kingdom: {col}')
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.savefig(f'residual_by_kingdom_{col}.png')
        plt.close()
        print(f"  Saved kingdom residual plot: {col}")

    # Residual correlation heatmap
    residual_cols = [f'{col}_residual' for col in final_cols]
    plt.figure(figsize=(8, 6))
    sns.heatmap(results_df[residual_cols].corr(),
                annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
    plt.title('Residual Correlation Between Targets')
    plt.tight_layout()
    plt.savefig('residual_correlation_heatmap.png')
    plt.close()
    print("  Saved residual correlation heatmap.")

    # Residual summary table
    print("\n" + "="*55)
    print("RESIDUAL SUMMARY")
    print("="*55)
    print(f"{'Target':<22} {'Mean':>8} {'Std':>8} {'Min':>8} {'Median':>8} {'Max':>8}")
    print("-"*55)
    for col in final_cols:
        r = results_df[f'{col}_residual'].dropna()
        print(f"  {col:<20} {r.mean():>8.3f} {r.std():>8.3f} "
              f"{r.min():>8.3f} {r.median():>8.3f} {r.max():>8.3f}")

except Exception as e:
    print(f"Error during residual analysis: {e}")
    import traceback
    traceback.print_exc()

print("\nAttention-Based Multi-Output Forecaster — completed successfully.")


RESIDUAL ANALYSIS
  Saved residual plot: Avg_Temperature
  Saved residual plot: Radiation
  Saved residual plot: Rain_Amount
  Saved residual plot: Wind_Speed
  Saved residual plot: Wind_Direction
  Saved kingdom residual plot: Avg_Temperature
  Saved kingdom residual plot: Radiation
  Saved kingdom residual plot: Rain_Amount
  Saved kingdom residual plot: Wind_Speed
  Saved kingdom residual plot: Wind_Direction
  Saved residual correlation heatmap.

RESIDUAL SUMMARY
Target                     Mean      Std      Min   Median      Max
-------------------------------------------------------
  Avg_Temperature        -0.045    0.483   -3.909   -0.035    3.227
  Radiation              -0.213    1.107  -11.470   -0.127    3.973
  Rain_Amount             0.464    7.737  -38.546    0.000  170.156
  Wind_Speed              0.089    3.007  -17.290   -0.042   20.988
  Wind_Direction          1.873   72.440 -319.358    1.610  331.794

Attention-Based Multi-Output Forecaster — completed successful